# Task 17 — DataFrame Practice and IPL Analysis

This task notebook is a practical worksheet that takes everything you learned about the pandas DataFrame in the previous sessions and puts it to work. It has two clear halves. The first half (Q-1 to Q-4) plays with a small handmade DataFrame about eleven birds: their age, how many times they were visited, and whether they were a priority. Because the data is tiny, it is the perfect laboratory for the core DataFrame operations — constructing a table from a dictionary, asking pandas for basic information, pulling out alternate rows, selecting rows and columns together, filtering with conditions, spotting missing values, filling NaN with a statistic, and removing duplicate rows. Every one of these operations shows up again in real projects, so the small data lets you see exactly what each call does before the stakes get higher.

The second half (Q-5 to Q-10) jumps to a serious dataset: every IPL match from 2008 to 2022. Here the questions are the kind you will actually be asked in interviews and on the job. Q-5 cleans team names because some franchises changed names over the years. Q-6 builds a bar chart of the five teams with the most matches, which requires counting each team across two columns at once. Q-7 finds the player who won the most player-of-the-match awards while playing against Mumbai Indians. Q-8 asks you to write a reusable function that takes any two team names and reports their head-to-head win-loss record plus the most frequent player of the match between them. Q-9 finds the top seven cities where the Kolkata Knight Riders play most often and plots them. Q-10 computes the average winning margin for Mumbai Indians in the 2011 season.

Work the questions strictly in order. Each one builds muscle the next one needs: the boolean masks from Q-2 and Q-3 feed the aggregations of Q-7 and Q-8; value_counts and drop_duplicates from Q-4 reappear constantly; and the team-name cleaning in Q-5 must happen before any of the later counting can be trusted. When you print a result, look at it — the counts in Q-6 and Q-7 depend on the data being cleaned correctly in Q-5.

There is no single correct style of answer; any correct pandas code is acceptable. The goal is fluency: reaching for the right method without hesitation. Predict each output before you run it, and compare your answer with what the notebook prints.

In [ ]:
"https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"

# 1. The birds DataFrame

## 1.1 A dictionary that becomes a table

The whole first exercise rests on one Python dictionary and one list:

```python
data = {'birds': ['Cranes', 'Cranes', 'plovers', 'spoonbills', 'spoonbills', 'Cranes', 'plovers', 'Cranes', 'spoonbills', 'spoonbills', 'Cranes'],
        'age': [3.5, 4, 1.5, np.nan, 6, 3, 5.5, np.nan, 8, 4, 3.5],
        'visits': [2, 4, 3, 4, 3, 4, 2, 2, 3, 2, 2],
        'priority': ['yes', 'yes', 'no', np.nan, 'no', 'no', 'no', 'yes', 'no', 'no', 'yes']}
labels = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k']
```

A dictionary of lists is the most readable way to describe a table: every key becomes a column name, and the list under it becomes that column's values. All four lists must have the same length, because pandas pairs them up row by row — here that means eleven rows and four columns.

Look closely at the values, because the questions are built on them. The `age` column contains two `np.nan` entries — missing data deliberately planted so that later questions can practise filling holes. The `priority` column has one `np.nan` too. `birds` holds four distinct categories, three of them repeated, which sets up the value_counts and duplicate questions. And `labels` is the index you can attach with the `index=` argument when constructing the frame.

The mental model is the whole point of a DataFrame: a row-and-column grid, each column a Series, all columns sharing the same row labels. Building one from a dictionary is the construction route you will use again and again.

### Q-1: Create the DataFrame and inspect it

i. Create a DataFrame `birds` from the dictionary `data` above, using `labels` as the index.
ii. Display basic information about the DataFrame.
iii. Show alternate rows of the DataFrame.

Behind the three sub-questions is the standard first-contact ritual with any table. Part (i) is construction: `pd.DataFrame(data, index=labels)` takes the dictionary, uses each key as a column, and attaches the eleven labels as the row index — row 'a' becomes the first Cranes record, row 'k' the last.

Part (ii) is the health check. `birds.info()` prints the index range, the total number of columns, and a per-column summary — how many values are NOT missing and what dtype each column holds. This is where you see at a glance that `age` has 9 non-null of 11, `priority` has 10 of 11, and that pandas used float64 for `age` (because of the NaN values) and object for the text columns. `birds.describe()` (a good extra) reports count, mean, min, max, and percentiles for the numeric columns.

Part (iii) is slicing. 'Alternate rows' means taking every second row. The cleanest idioms are `birds.iloc[::2]` (start at the top, step by 2) and `birds[::2]`, both of which return rows a, c, e, g, i, k. You could also hand-pick rows with `birds.iloc[[0, 2, 4, 6, 8, 10]]` — more explicit, same answer.

> Hint: `pd.DataFrame(data, index=labels)` for (i), `birds.info()` for (ii), `birds.iloc[::2]` or `birds[::2]` for (iii).

In [ ]:
# code here

### Q-2: Row and column selection

i. Show only rows [1st, 3rd, 7th] from columns ['bird', 'age'].
ii. Select rows where the number of visits is less than 4.

Part (i) combines row selection with column selection in one expression. The rows are identified by name — the 1st, 3rd, and 7th rows of the birds frame are the labels 'a', 'c', and 'g' — so label-based access makes the intent explicit: `birds.loc[['a', 'c', 'g'], ['birds', 'age']]`. (Note the question spells the column 'bird'; in your DataFrame the column is actually called 'birds', so use whichever name your table really has.) The result is a small 3 by 2 DataFrame containing exactly the cells that match both the row labels and the column list. The same trick with positions would be `birds.iloc[[0, 2, 6], [0, 1]]` — row positions 0, 2, and 6, and the first two columns.

Part (ii) is boolean filtering — the single most important pandas pattern. Build a mask with `birds['visits'] < 4`, then place it inside brackets: `birds[birds['visits'] < 4]`. The output keeps every row whose visits count is below four. Each condition produces one True/False per row, and pandas keeps only the True rows, index labels intact.

> Hint: `.loc[['a','c','g'], ['birds','age']]` for (i) and `birds[birds['visits'] < 4]` for (ii).

In [ ]:
# code here

### Q-3: Missing values

i. Select all rows with nan values in the age and visits columns.
ii. Fill nan with the respective series mode value.

Part (i) asks for rows where age OR visits is missing — in this small table only the `age` column actually contains NaN, so the answer is the two rows labelled 'd' and 'h'. Detection always starts with `isnull()`: `birds['age'].isnull()` returns a mask, and `birds[birds['age'].isnull() | birds['visits'].isnull()]` keeps the rows flagged on either column. The `|` is the logical OR for pandas masks, and each condition must be wrapped in parentheses, exactly as with ordinary multi-condition filters.

Part (ii) wants the mode — the most frequent value of the column — pushed into each hole. `birds['age'].mode()` returns a Series of the most common age(s), so `birds['age'].mode()[0]` takes the first. Then `birds['age'].fillna(birds['age'].mode()[0], inplace=True)` stamps every missing age with that value. Because `inplace=True` runs inside the function, the change is permanent, and a later print of `birds` shows no NaNs left in the age column. The same recipe works column by column — the mode of the categorical `priority` column would be the most frequent 'no'.

> Hint: combine masks with `isnull() | ...` for (i); `fillna(birds['age'].mode()[0], inplace=True)` for (ii).

In [ ]:
# code here

### Q-4: Totals and duplicates

i. Find the total number of visits of the bird Cranes.
ii. Find the number of each type of birds in the dataframe.
iii. Print the number of duplicate rows.
iv. Drop duplicate rows and make this change permanent; show the dataframe afterwards.

Part (i) is a filter-then-aggregate pipeline: `birds[birds['birds'] == 'Cranes']['visits'].sum()` — keep only the Cranes rows, take the visits column, and add the values. Read it left to right and it reads like English.

Part (ii) is a frequency table: `birds['birds'].value_counts()` counts how many of each bird type appear and sorts descending — three distinct species, each counted.

Part (iii) asks how many rows are exact repeats. `birds.duplicated()` returns a boolean mask marking every duplicate row, and `.sum()` counts the True entries — in this frame there is one duplicate row, which you can also see by eye: two identical spoonbills-age-6 rows.

Part (iv) deletes the repeats: `birds.drop_duplicates(inplace=True)` removes all but the first occurrence of each duplicated row. Because inplace changes the object itself, the table printed afterwards has exactly ten rows instead of eleven. `keep='first'` (the default) keeps the earliest copy; `keep='last'` would keep the newest.

> Hint: chain a mask with `.sum()` for (i), call `.value_counts()` for (ii), count `.duplicated().sum()` for (iii), then `drop_duplicates(inplace=True)` for (iv).

In [ ]:
# code here

# 5. The IPL Matches Dataset

The second half of the notebook switches from eleven birds to a serious real-world table: the complete IPL match history from 2008 to 2022. The dataset is available from the provided Kaggle page and Google Drive link — either download the file `IPL_Matches_2008_2022.csv` and read it with `pd.read_csv`, or load it directly from a URL if one is given.

```python
ipl = pd.read_csv('/content/IPL_Matches_2008_2022.csv')
```

Each row is one IPL match from 2007/08 to 2021. The columns you will lean on constantly include `Team1` and `Team2` (the two sides), `Winner` / `WinningTeam`, `City` and `Venue` (where it was played), `Player_of_Match`, `Margin` (the runs or wickets won by), `Season` (with entries like '2022', '2021', '2020/21', '2009/10'), and `Result`. Two columns hold the teams and one column the winner — that is exactly the kind of structure the questions want you to think about carefully.

Before any counting can be trusted, the data must be made consistent. Three franchises renamed themselves during the period the file covers: 'Delhi Daredevils' became 'Delhi Capitals', 'Kings XI Punjab' became 'Punjab Kings', and 'Rising Pune Supergiants' became 'Rising Pune Supergiant'. If you count teams without unifying these names, your answers will silently treat one franchise as two different teams.

One deliberate trap: 'Gujarat Titans' and 'Gujarat Lions' look similar but are genuinely different teams — never merge them. The `.replace()` method on the two team columns is the natural tool for the renaming.

> Hint: after reading the file, replace the old names on BOTH `Team1` and `Team2` before any counting question.

### Q-5: Rename the teams

In the IPL dataset some team names have changed. You will have to consider them as the same:

```
'Delhi Capitals' formerly as 'Delhi Daredevils'
'Punjab Kings' formerly as 'Kings XI Punjab'
'Rising Pune Supergiant' formerly as 'Rising Pune Supergiants'
```

Make the changes accordingly, using the current name for each team. Be careful: Gujarat Titans and Gujarat Lions are different teams.

Column-wise renaming is the tool: `ipl['Team1'] = ipl['Team1'].replace({'Delhi Daredevils': 'Delhi Capitals', 'Kings XI Punjab': 'Punjab Kings', 'Rising Pune Supergiants': 'Rising Pune Supergiant'})` — and the identical call for `Team2`. The `replace` method takes a dictionary mapping old values to new ones and swaps every occurrence in the column. Two columns, two calls.

Why bother? Every counting question that follows depends on clean names. Q-6 counts matches per team, Q-7 filters against Mumbai Indians, and Q-8 matches two team names — if any of the old names survive, those teams are undercounted. Verify your work with `ipl['Team1'].value_counts()` and `ipl['Team2'].value_counts()`: after renaming, 'Delhi Capitals', 'Punjab Kings', and 'Rising Pune Supergiant' should show bigger counts than before, no old name should remain anywhere, and the Gujarat Titans / Gujarat Lions pair should still exist as two separate rows.

> Hint: two `.replace({...})` calls (one per team column), then confirm with value_counts.

In [ ]:
# code here

### Q-6: Top five teams by matches

Write a code which can display the bar chart of the top 5 teams who have played the maximum number of matches in the IPL.

> Hint: Be careful — the data is divided into 2 different columns (Team 1 and Team 2).

Every match involves two teams, and the file stores them in two separate columns. A team's true match count is its appearances in `Team1` PLUS its appearances in `Team2`. The cleanest way to assemble a per-team count across both columns is to concatenate them into one Series and count once:

```python
all_teams = pd.concat([ipl['Team1'], ipl['Team2']])
counts = all_teams.value_counts()
```

`value_counts()` then counts how often each franchise appears on either side. Taking `.head(5)` keeps the five most frequent teams, and `.plot(kind='bar')` renders them as a bar chart — each team a bar, its match count the bar height. `pd.concat` stacks the two Series on top of each other, which is precisely the 'put both team columns together first' idea the hint is pointing at. An alternative that also works is `ipl['Team1'].value_counts() + ipl['Team2'].value_counts()` — adding the two frequency tables element-wise — just make sure the renaming from Q-5 ran first, or both tables misalign.

> Hint: concat the two team columns, value_counts, head(5), then plot(kind='bar').

In [ ]:
# code here

### Q-7: Most player-of-the-match awards against Mumbai Indians

Find the player who won the most player-of-the-match awards while playing AGAINST the Mumbai Indians.

> Just for this question assume the player-of-the-match award is given to a player from the winning team (which is true in most cases).

Think about which condition the award depends on. A match counts if the Mumbai Indians are one of the two teams, AND the winner is not the Mumbai Indians — because the award goes to a player of the winning side, and the question wants players who performed against them. The mask therefore has two parts joined by `&`:

```python
matches = ipl[((ipl['Team1'] == 'Mumbai Indians') | (ipl['Team2'] == 'Mumbai Indians')) &
              (ipl['WinningTeam'] != 'Mumbai Indians')]
```

Every parenthesised condition matters: `|` covers matches where Mumbai are Team1 or Team2, and `!= 'Mumbai Indians'` excludes the matches they won — a Mumbai player cannot be 'playing against' Mumbai. Once the frame is filtered, the leaderboard is one call: `matches['Player_of_Match'].value_counts()` sorts award winners by frequency, and `.head(1)` reveals the top name.

> Hint: filter with `&` over the two conditions, then `value_counts().head(1)`.

In [ ]:
# code here

### Q-8: Team1 vs Team2 dashboard

Create a function that takes two strings (names of two teams) as input. Show the win-loss record between them and the player getting the most player-of-the-match awards in matches between these two teams.

```
team1_vs_team2('Kolkata Knight Riders', 'Chennai Super Kings')
```

This question asks for a reusable dashboard — a function, not a one-off query. Start by defining `def team1_vs_team2(team_1, team_2):`. Inside it, the first job is isolating head-to-head matches: a match is between the two teams when one appears in each column,

```python
matches = ipl[((ipl['Team1'] == team_1) & (ipl['Team2'] == team_2)) |
              ((ipl['Team1'] == team_2) & (ipl['Team2'] == team_1))]
```

The `|` covers both orderings — the two teams meet once with each as Team1.

The win-loss record needs the winner information: `matches['WinningTeam'].value_counts()` counts how often each side won between them. For the player-of-the-match honour, `matches['Player_of_Match'].value_counts().head(1)` gives the player who starred most often in their fixtures. The function can print both results, or return a tuple of (record, top player) for further use. Try it on the example pair, then on any pair you like — the function should behave identically for every input.

> Hint: mask with both orders joined by `|`; value_counts for wins and for player of the match.

In [ ]:
# code here

### Q-9: Cities where the Kolkata Knight Riders play most

Find the top 7 cities where the matches of the Kolkata Knight Riders are played frequently and plot the result as a bar chart.

> `.plot(kind = 'bar')` can help you plot the bar chart.

Two-sided reasoning again: KKR are present when they appear as Team1 or as Team2. To focus on venues, filter first, then count by city:

```python
kkr = ipl[(ipl['Team1'] == 'Kolkata Knight Riders') | (ipl['Team2'] == 'Kolkata Knight Riders')]
kkr['City'].value_counts().head(7).plot(kind='bar')
```

The pipeline reads left to right: filter the KKR matches, count how many fell in each city, keep the top seven, and draw them as a bar chart. All of KKR's away and home fixtures are included because both team columns were checked. `.head(7)` is the limiter the question names, and the tallest bar is the city where they have played most often.

> Hint: filter KKR matches (Team1 or Team2), then `City` value_counts, head(7), plot(kind='bar').

In [ ]:
# code here

### Q-10: Average margin for the Mumbai Indians in 2011

Find the average margin for the team Mumbai Indians for only the season 2011.

'Season 2011' is a straightforward filter: `ipl[ipl['Season'] == '2011']`. Within those rows, focus on matches the Mumbai Indians played and won — the margin column only makes sense for the winning team. So combine the conditions:

```python
mi_2011 = ipl[(ipl['Season'] == '2011') &
              ((ipl['Team1'] == 'Mumbai Indians') | (ipl['Team2'] == 'Mumbai Indians'))]
mi_2011['Margin'].mean()
```

`mi_2011['Margin'].mean()` averages the margins of all those games. Depending on the column semantics in your file the margin may be a number of runs; if the file mixes units you may need to inspect `mi_2011[['Margin', 'Margin_Type']]` first. The mean is the final answer — a single number summarising how strongly the Mumbai Indians typically won that season.

> Hint: filter Season == '2011' with a Mumbai team mask, then `Margin.mean()`.

## Summary

Task 17 took you through the complete DataFrame workflow on a real IPL dataset. In the first half you built a DataFrame from a dictionary — birds, age, visits, priority — and inspected it with info(), sliced it for alternate rows and specific row-column windows, filtered with boolean masks, and handled missing values by filling NaN with the mode. You totalled the visits of one bird, counted categories with value_counts, and removed duplicate rows permanently with drop_duplicates. In the second half you cleaned real franchise names with replace, counted matches across two columns and plotted the top five, answered award and win-loss questions with filtered value_counts, wrote a reusable two-team dashboard function, found the Kolkata Knight Riders' busiest cities, and computed the Mumbai Indians' average margin in 2011. The through-line is the same everywhere: clean the names, filter to the rows that matter, count, and aggregate.

In [ ]:
# code here